# Hour 1 — Login & Your First ClearML Experiment

In this hour you will:
1. Connect this notebook to the workshop ClearML server with **your own** API keys.
2. Run your first tracked experiment with **two lines of code**.
3. Do quick EDA on the plant-disease dataset and watch the plots get **auto-captured** into ClearML.

Everything you see here shows up live in the Web UI at **http://202.131.110.56**.

## 1. Connect to the ClearML server

Everyone shares the **same** ClearML login + API key (given to you by the facilitator) — paste it below. You tell your work apart with **`WORKSHOP_USER`**: set it to *your name*, and all your experiments land in your own project **`PlantVillage Workshop/<your-name>`**.

> Run this cell **first**, before any `from clearml import ...`.

In [ ]:
import os
# --- SHARED credentials (same for everyone). Paste the key + secret between the quotes. ---
# (plain os.environ, NOT %env: an empty '' really is empty, and these # comments stay comments)
os.environ['CLEARML_WEB_HOST']       = 'http://202.131.110.56'
os.environ['CLEARML_API_HOST']       = 'http://202.131.110.56:8008'
os.environ['CLEARML_FILES_HOST']     = 'http://202.131.110.56:8081'
os.environ['CLEARML_API_ACCESS_KEY'] = ''   # <-- paste the SHARED access key here
os.environ['CLEARML_API_SECRET_KEY'] = ''   # <-- paste the SHARED secret key here
os.environ['WORKSHOP_USER']          = ''   # <-- YOUR name, e.g. 'alice' (letters/dashes, no spaces)

In [ ]:
# Colab already has torch/numpy/pandas/matplotlib/seaborn/scikit-learn/pillow.
# The heavy TRAINING stack runs on the agent (not here), so we only add two libs:
#   clearml      -> talk to the ClearML server (all hours)
#   transformers -> only needed for Hour 4's local inference (harmless to install now)
#   python-dotenv -> lets the next cell fall back to a .env file if you leave the creds blank
!pip install -q clearml transformers python-dotenv

## 2. Your first experiment — two lines

`Task.init()` opens an experiment and immediately starts auto-capturing your code, notebook source, installed packages, environment, console output, and any plots/metrics you produce.

In [ ]:
import os
from clearml import Task

# Normalize: strip stray quotes/whitespace so a value like '""' counts as EMPTY.
for _k in ('CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY', 'WORKSHOP_USER',
           'CLEARML_WEB_HOST', 'CLEARML_API_HOST', 'CLEARML_FILES_HOST'):
    _v = os.environ.get(_k)
    if _v is not None:
        os.environ[_k] = _v.strip().strip('"').strip("'").strip()

# If creds/name are still blank, fall back to a .env file (override=True replaces the empties).
if not (os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('WORKSHOP_USER')):
    try:
        from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv(usecwd=True), override=True)
    except Exception:
        pass

assert os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('CLEARML_API_SECRET_KEY'), \
    'ClearML API keys are EMPTY — paste the shared key into the credentials cell (or add a .env), then re-run.'
assert os.environ.get('WORKSHOP_USER'), 'Set WORKSHOP_USER (your name) in the credentials cell or .env.'

WORKSHOP_USER = os.environ['WORKSHOP_USER']
PROJECT = f'PlantVillage Workshop/{WORKSHOP_USER}'   # your own project
Task.force_store_standalone_script()   # store code standalone (no git clone needed by the agent)
task = Task.init(project_name=PROJECT, task_name='EDA')
print('Your project:', PROJECT)
print('Task page:', task.get_output_log_web_page())

In [ ]:
# Log a few hyperparameters — these become editable in the UI (used in Hour 3).
params = task.connect({
    'dataset_project': 'PlantVillage',
    'full_dataset': 'plantdisease_full',
    'demo_dataset': 'plantdisease_demo',
})
params

## 3. EDA — class distribution (auto-captured)

The dataset is **shared** — one common copy on the server (project `PlantVillage`) that every attendee uses. We read its **file list** (no image download) and count images per class. The bar chart is captured automatically the moment we call `plt.show()`. `alias=` binds the dataset to this task so the UI records which data you looked at.

In [ ]:
from clearml import Dataset
import pandas as pd

# The full catalog if the facilitator registered it, else fall back to the demo subset.
try:
    ds = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_full', alias='plantdisease_full')
except Exception:
    ds = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_demo', alias='plantdisease_demo')
files = ds.list_files()                         # metadata only, no image download
classes = [f.split('/')[0] for f in files]      # first path segment = class folder
counts = pd.Series(classes).value_counts().sort_values(ascending=False)
print(f'{len(files)} images across {counts.size} classes')
counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 5))
sns.barplot(x=counts.values, y=counts.index, color='#4C9A2A')
plt.title('PlantVillage — images per class')
plt.xlabel('image count'); plt.ylabel('')
plt.tight_layout()
plt.show()   # <-- auto-captured into the task's PLOTS tab

In [ ]:
# Report the same distribution as an interactive table in ClearML.
logger = task.get_logger()
logger.report_table('Class distribution', 'per class', iteration=0,
                    table_plot=counts.rename('count').reset_index().rename(columns={'index': 'class'}))
# A couple of single values land in the SCALARS 'summary' section.
logger.report_single_value('total_images', len(files))
logger.report_single_value('num_classes', int(counts.size))

## 4. EDA — a grid of sample leaves (in DEBUG SAMPLES)

We pull the small **demo** dataset (~1000 images, 4 classes) and show a few examples. Image grids can't render as interactive Plotly plots, so we report the figure as a **PNG** (`report_image=True`) → it appears in the **DEBUG SAMPLES** tab (not PLOTS).

In [ ]:
import os, random
from PIL import Image

demo_dir = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_demo', alias='plantdisease_demo').get_local_copy()
demo_classes = sorted(d for d in os.listdir(demo_dir) if os.path.isdir(os.path.join(demo_dir, d)))
print('demo classes:', demo_classes)

random.seed(0)
fig = plt.figure(figsize=(12, 3 * len(demo_classes)))
for r, cls in enumerate(demo_classes):
    imgs = os.listdir(os.path.join(demo_dir, cls))
    for c, name in enumerate(random.sample(imgs, 4)):
        ax = fig.add_subplot(len(demo_classes), 4, r * 4 + c + 1)
        ax.imshow(Image.open(os.path.join(demo_dir, cls, name)).convert('RGB'))
        ax.axis('off')
        if c == 0:
            ax.set_ylabel(cls, fontsize=9)
        ax.set_title(cls, fontsize=8)
plt.tight_layout()
# imshow grids can't render as Plotly plots, so report the figure as a PNG (report_image=True)
# -> it appears in the DEBUG SAMPLES tab (not PLOTS).
task.get_logger().report_matplotlib_figure('Sample leaves', 'per class', iteration=0, figure=fig, report_image=True)
plt.show()

## 5. Done — go look at the UI

Open your task at **http://202.131.110.56** → project **`PlantVillage Workshop/<your-name>`**. You'll find:
- **PLOTS**: the class-distribution bar chart
- **DEBUG SAMPLES**: the sample-leaf grid
- **SCALARS → summary**: total_images / num_classes
- **CONFIGURATION → HYPERPARAMETERS**: the params you connected
- **EXECUTION**: your notebook source, packages, and environment — all captured automatically.

Next up (Hour 2): send a real training job to the shared GPU queue.